# Цифровые технологии в профессиональной деятельности
## Лабораторная работа № 2. Сетевой анализ: персонажи «Ведьмака»

**Дата:** 30.05.2026

**Сдача:** заполненную тетрадку отправьте через форму до 23:59 сегодняшнего дня.

---

На семинаре мы строили граф по генеалогическому датасету. Теперь работаем с **сетью взаимодействий персонажей** в книгах Анджея Сапковского.

Датасет построен по принципу co-occurrence: если два персонажа упоминаются в одной сцене, между ними фиксируется связь. Вес связи — количество таких совместных упоминаний.


In [1]:
# Запустите эту ячейку в первую очередь

# pip install kagglehub[pandas-datasets]

import pandas as pd
import networkx as nx
from pyvis.network import Network
from IPython.display import IFrame, display

import kagglehub
from kagglehub import KaggleDatasetAdapter

---
## Часть 1. Загрузка и изучение данных

In [2]:
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "avasadasivan/witcher-network",
  "witcher_network.csv",
)

C:\Users\NYX\AppData\Local\Temp\ipykernel_14912\2772197647.py:1: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


**Задание 1.1.** Загрузите файл `witcher_small_network.csv` в датафрейм `df`. Выведите первые 10 строк.

In [3]:
# ВАШ КОД


**Задание 1.2.** Посмотрите на данные и ответьте на вопросы:
- Сколько уникальных персонажей в датасете?
- Сколько книг охватывает датасет?

In [4]:
# Подсказка: уникальные персонажи — это объединение столбцов Source и Target


**Ваш ответ:** *(напишите здесь)*

---
## Часть 2. Подготовка данных: агрегация рёбер

Прежде чем строить граф, обратите внимание на структуру данных. Найдите в датафрейме пару Geralt и Yennefer:

In [5]:
# Смотрим на все строки, где фигурируют Geralt и Yennefer
mask = (
    ((df['Source'] == 'Geralt') & (df['Target'] == 'Yennefer')) |
    ((df['Source'] == 'Yennefer') & (df['Target'] == 'Geralt'))
)
df[mask]

,Unnamed: 0,Source,Target,Type,Weight,book
207,207,Geralt,Yennefer,Undirected,22,1
222,222,Yennefer,Geralt,Undirected,23,1
265,265,Yennefer,Geralt,Undirected,25,2
268,268,Geralt,Yennefer,Undirected,7,2
429,429,Geralt,Yennefer,Undirected,19,3
432,432,Yennefer,Geralt,Undirected,20,3
808,808,Geralt,Yennefer,Undirected,23,4
865,865,Yennefer,Geralt,Undirected,19,4
1305,1305,Yennefer,Geralt,Undirected,7,5
1422,1422,Geralt,Yennefer,Undirected,4,5


**На этом моменте необходимо понять, как собирались данные для датасета.**
Допустим, в тексте есть такой фрагмент (строки пронумерованы):
```
1: Геральт вошёл в таверну и огляделся.
2: За столом сидела Йенифер, не поднимая глаз.
3: На улице шумел дождь.
4: Данделион допивал своё вино в углу.
5: Геральт подошёл и сел напротив неё.
6: Йенифер наконец взглянула на него.
7: Данделион заметил их и помахал рукой.
8: За окном проехала телега.
```
Скрипт читает текст построчно. Находит в строке 1 имя Геральт — это Source. Дальше смотрит в окно ±5 строк (строки 1–6). В этом окне встречаются Йенифер (строка 2) и Данделион (строка 4). Записывает два ребра: Geralt-Yennefer +1 и Geralt-Dandelion +1.

Потом доходит до строки 2, находит Йенифер — теперь она Source. Окно: строки 1–7. Встречает Геральт (строка 1), Данделион (строки 4 и 7). Записывает: Yennefer-Geralt +1 и Yennefer-Dandelion +1.

В строке 4 находит Данделион. Окно: строки 1–8 (но там строк меньше — берёт что есть). Встречает Геральт, Йенифер, снова Йенифер и Данделион... то есть снова Геральт и Йенифер. Dandelion-Geralt +1, Dandelion-Yennefer +2 (два упоминания в окне).

В итоге после этого крошечного фрагмента в таблице будет:
```
Geralt    - Yennefer  : 1
Geralt    - Dandelion : 1
Yennefer  - Geralt    : 1
Yennefer  - Dandelion : 1
Dandelion - Geralt    : 1
Dandelion - Yennefer  : 2
```
Отсюда и несимметричность: Данделион встречает Йенифер дважды в своём окне, а Йенифер встречает Данделиона один раз в своём.

**Вторая проблема**: двойные связи Geralt-Yennefer и Yennefer-Geralt указываются отдельно в каждой книге.

Поскольку граф **неориентированный** (связь симметрична), нам нужно:
1. Привести все пары к единому порядку: `(A, B)` и `(B, A)` -> одна пара `(меньший, больший)` по алфавиту
2. Просуммировать веса всех строк с одинаковой парой — получим суммарный вес связи по всем книгам

In [7]:
# Шаг 1. Нормализуем порядок в паре: всегда (алфавитно меньший, алфавитно больший)
# sorted([A, B]) возвращает список из двух элементов в алфавитном порядке
df['node_1'] = df.apply(lambda row: sorted([row['Source'], row['Target']])[0], axis=1)
df['node_2'] = df.apply(lambda row: sorted([row['Source'], row['Target']])[1], axis=1)

In [8]:
# Шаг 2. Агрегируем: суммируем веса по уникальным парам
edges = (
    df.groupby(['node_1', 'node_2'], as_index=False)['Weight']
    .sum()
    .rename(columns={'node_1': 'source', 'node_2': 'target', 'Weight': 'weight'})
)

In [10]:
print(f"Строк до агрегации: {len(df)}")
print(f"Уникальных рёбер после агрегации: {len(edges)}")
print()

Строк до агрегации: 2600
Уникальных рёбер после агрегации: 1267



**Задание 2.1.** Выведите топ-10 самых частотных рёбер

---
## Часть 3. Построение взвешенного графа

**Задание 3.1.** Постройте взвешенный неориентированный граф `G` из таблицы `edges`. Добавьте рёбра с атрибутом `weight`.

*Подсказка:* `G.add_edge(source, target, weight=w)`

In [ ]:
G = nx.Graph()

for _, row in edges.iterrows():
    # ВАШ КОД: добавьте ребро с весом
    pass

print(f"Узлов: {G.number_of_nodes()}")
print(f"Рёбер: {G.number_of_edges()}")

**Задание 3.2.** Вычислите плотность графа. Это плотная или разреженная сеть? Что это означает для взаимодействий персонажей?

In [ ]:
# ВАШ КОД


**Ваш ответ:** *(напишите здесь)*

---
## Часть 4. Метрики центральности

В семинарной тетрадке мы считали метрики для ориентированного графа. Сейчас граф **неориентированный**, поэтому используем `nx.degree_centrality()` и `nx.betweenness_centrality()` без разделения на `in/out`.

**Задание 4.1.** Вычислите `degree_centrality` и `betweenness_centrality` для всех узлов. Выведите результаты в виде двух отсортированных таблиц (от большего к меньшему).

In [ ]:
# ВАШ КОД


**Задание 4.2.** Сравните два рейтинга. Совпадают ли лидеры? Есть ли персонаж, который высоко по betweenness, но не является самым «популярным» по degree? Если да — объясните, что это означает для его роли в нарративе.

**Ваш ответ:** *(напишите здесь)*

---
## Часть 5. Визуализация

**Задание 5.1.** Постройте интерактивную визуализацию графа с помощью PyVis со следующими условиями:

- **Размер узла** зависит от degree centrality: `size = 10 + degree_centrality[node] * 80`
- **Толщина ребра** (`width`) равна логарифму веса: `width = weight ** 0.5` — это сглаживает разброс между слабыми и сильными связями
- **Всплывающая подсказка** (`title`) содержит имя персонажа и его degree centrality
- Сохраните файл как `witcher_network.html` и отобразите в тетрадке

In [ ]:
# Сначала записываем метрики как атрибуты узлов в граф

for node in G.nodes():

# Записываем толщину рёбер
for u, v, data in G.edges(data=True):

In [ ]:
# ВАШ КОД: создайте Network, импортируйте граф, сохраните и отобразите


**Задание 5.2.** Посмотрите на получившуюся визуализацию. Какие кластеры или группы персонажей вы видите? Соответствует ли структура сети вашим знаниям о книгах (или фильме/сериале)? Напишите 3–4 предложения.

**Ваш ответ:** *(напишите здесь)*

---
## Дополнительное задание (+2 балла)

Датасет содержит столбец `book` — номер книги, в которой зафиксировано взаимодействие, это позволяет изучить, как **меняется структура сети** от книги к книге.

**Ваша задача:**

1. Выберите **две любые книги** (например, книгу 1 и книгу 7 — начало и конец).
2. Для каждой книги отдельно: отфильтруйте строки датафрейма `df`, агрегируйте рёбра (как в Части 2), постройте граф.
3. Для каждого графа выведите: число узлов, число рёбер, плотность, топ-3 персонажа по `degree centrality`.
4. Сравните результаты. Как изменился состав «главных» персонажей? Как изменилась плотность сети? Чем вы это объясняете?

*Подсказка:* `df_book1 = df[df['book'] == 1]`

In [ ]:
# ВАШ КОД


**Ваш ответ:** *(напишите здесь)*